# Aggregate by continent

The fleet's per-tile partial sums → the continents cube: runoff-onset statistics per continent × 1° latitude ×
100 m elevation × CHILI insolation class × water year, plus the GTOPO30 land-pixel histogram as `dem_pixel_count`.
Run it after the *Process tiles to parquets* workflow and before the other notebooks in this folder.

| | |
| --- | --- |
| Reads | `partials/<version>/tile_*.parquet`, downloaded from Azure (`snowmelt_runoff_onset_analysis/partials/<version>/`) when the SAS token is available, otherwise the local cache as is; `data/gtopo30_lat_elev_histogram.nc` (tracked; `pixi run gtopo30` rebuilds it from Earth Engine) |
| Writes | `data/aggregation/<version>/all_continents_<filter>.nc`, one cube per pixel filter; optionally the latitude × elevation × aspect cube `all_continents_aspect_<filter>.nc` |
| Needs | the Azure SAS token only for the partials download (the CI smoke test runs this notebook on two fixture tiles without any credential) |

What a partials row is, why sums are enough, and what changed against the 2025 workflow: `pipeline/README.md`
and `docs/aggregation_lineage.md`.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from gsro_analysis import aggregate, paths, settings

In [ ]:
config = settings.load_config()          # the dataset version lives in settings.CONFIG_FILE
VERSION = config.version
WATER_YEARS = [int(y) for y in config.water_years]
UNIT = 'continents'
FILTER_TAGS = list(aggregate.FILTERS)    # the pixel filters the fleet emitted: 'full_dataset' and 'fcf_lte_50' (the analyses' rule)
BUILD_ASPECT_CUBE = False                # also write the continent x latitude x elevation x aspect cube (~120 MB; flat pixels drop out)
aggregation_dir = paths.aggregation_dir(UNIT, VERSION)   # analyses/continents/data/aggregation/<version>/

# the Azure SAS token is needed for the partials download; without it (the CI smoke test on the
# fixture tiles) the notebook keeps going: using the local partials cache as is
try:
    config.sas_token
    HAVE_AZURE = True
except ValueError as e:
    HAVE_AZURE = False
    print(f'no Azure SAS token: using the local partials cache as is ({e})')
print(f'{VERSION} | water years {WATER_YEARS[0]}-{WATER_YEARS[-1]} | filters {FILTER_TAGS} | Azure: {HAVE_AZURE}')
print(f'cubes -> {aggregation_dir}')

## 1. The fleet's partial sums, one parquet per tile

Every row is one tile's contribution to one cell of the cube: the pixels of one (filter, unit type, unit id,
elevation / aspect / latitude bin, CHILI class) with their count and the sums the statistics need
(Σ median, Σ median², per water year Σ onset, Σ onset², Σ anomaly, Σ anomaly², the CHILI and forest-cover
correlation sums). A unit that spans several tiles is several rows; adding them is the reduce.

In [ ]:
partials_dir = paths.partials_cache(VERSION)                        # partials/<version>/ (gitignored)
if HAVE_AZURE:
    partial_files = aggregate.sync_partials(config, partials_dir)   # downloads the tiles missing from the cache, drops stale ones
else:
    partial_files = sorted(partials_dir.glob('tile_*.parquet'))
print(f'{len(partial_files)} tiles in {partials_dir}')

In [ ]:
tile_partials_df = pd.read_parquet(partial_files[0])
print(f'{partial_files[0].name}: {len(tile_partials_df)} rows x {len(tile_partials_df.columns)} columns; '
      'one row = one tile\'s pixels in one (filter, unit type, unit id, bins, CHILI class) cell')
tile_partials_df

In [ ]:
# Sum the partials over tiles, keeping only this unit type. The reduce is a sum over identical keys, so
# summing BATCH tiles at a time gives the same result as concatenating everything first, at a fraction of the
# memory (the full campaign is ~14 M rows; a mountain-range tile alone is ~10 k rows). Each tile is cut down to
# this unit before it joins the batch. min_count=1 keeps a column NaN when no tile reported it.
KEY_COLS = ['filter_tag', 'unit_type', 'unit_id', 'elevation', 'aspect', 'latitude', 'chili_class']
BATCH = 50
t0 = time.time()
summed_partials_df, n_rows = None, 0
for i in range(0, len(partial_files), BATCH):
    tiles = []
    for f in partial_files[i:i + BATCH]:
        tile_df = pd.read_parquet(f)
        if 'unit_type' not in tile_df.columns:    # a verified-empty tile (no pixel with a valid median passed the filters)
            continue
        tiles.append(tile_df[tile_df['unit_type'] == UNIT].drop(columns=['tile_row', 'tile_col'], errors='ignore'))
    batch_df = pd.concat(tiles, ignore_index=True)
    n_rows += len(batch_df)
    batch_sums_df = batch_df.groupby(KEY_COLS, sort=False, dropna=False).sum(min_count=1)
    del tiles, batch_df
    if summed_partials_df is None:
        summed_partials_df = batch_sums_df
    else:
        summed_partials_df = (pd.concat([summed_partials_df, batch_sums_df])
                              .groupby(level=KEY_COLS, sort=False, dropna=False).sum(min_count=1))
summed_partials_df = summed_partials_df.reset_index()
print(f'{n_rows:,} {UNIT} partial rows from {len(partial_files)} tiles summed into {len(summed_partials_df):,} '
      f'cube cells ({time.time() - t0:.0f}s)')
summed_partials_df

## 2. The land-area reference: GTOPO30 land pixels per latitude × elevation bin

The grey background of the continental panels is every land pixel, mapped or not: a count of GTOPO30 cells
per continent × 1° × 100 m bin, reduced once on Earth Engine (`pipeline/scripts/get_gtopo30_histogram.py`) and
tracked, since it does not depend on the dataset version.

In [ ]:
gtopo30_ds = xr.open_dataset(paths.gtopo30_histogram())
gtopo30_ds

## 3. The cube, one file per pixel filter

`aggregate.reduce_partials` turns the summed rows into bin means (Σx / n), standard deviations
(√(Σx² / n − mean²)), pixel counts and the CHILI / forest-cover correlations on the dense
`continent × latitude × elevation × chili_class × water_year` grid. The aspect key of the partials is summed out
for the default cube, so flat pixels (no aspect) still count; continents are named on the fixed six-continent
axis (Australia folded into Oceania, Antarctica dropped) so a partially processed version still runs.

In [ ]:
GROUPS = ['continents'] + (['continents_aspect'] if BUILD_ASPECT_CUBE else [])
for group in GROUPS:
    for filter_tag in FILTER_TAGS:
        t0 = time.time()
        try:
            cube_ds = aggregate.reduce_partials(summed_partials_df, group, filter_tag, WATER_YEARS)
        except ValueError as e:               # no rows for this filter (a partially processed version)
            print(f'{group}/{filter_tag}: {e}')
            continue
        # the GTOPO30 land-pixel count on the cube's own grid
        cube_ds['dem_pixel_count'] = gtopo30_ds['pixel_count'].reindex(
            continent=cube_ds['continent'], latitude=cube_ds['latitude'], elevation=cube_ds['elevation']).astype('int64')
        cube_ds['dem_pixel_count'].attrs = gtopo30_ds['pixel_count'].attrs
        cube_ds.attrs.update({'dataset_version': VERSION, 'n_tiles': len(partial_files),
                              'produced_by': 'analyses/continents/0_aggregate_by_continent.ipynb'})
        cube_path = aggregation_dir / f'all_{group}_{filter_tag}.nc'
        encoding = {v: {'zlib': True, 'complevel': 4, **({'dtype': 'float32'} if cube_ds[v].dtype.kind == 'f' else {})}
                    for v in cube_ds.data_vars}
        cube_ds.to_netcdf(cube_path.with_suffix('.nc.tmp'), encoding=encoding)
        cube_path.with_suffix('.nc.tmp').replace(cube_path)
        print(f'wrote {cube_path.name}: {cube_path.stat().st_size / 1e6:.1f} MB, dims {dict(cube_ds.sizes)} ({time.time() - t0:.0f}s)')

In [ ]:
continents_ds = xr.open_dataset(aggregation_dir / 'all_continents_fcf_lte_50.nc')
continents_ds

In [ ]:
# a first look: median runoff onset per continent, latitude and elevation, all CHILI classes together
median_onset_da = aggregate.collapse(continents_ds)['runoff_onset_median']
median_onset_da.plot(col='continent', col_wrap=3, x='elevation', y='latitude', vmin=100, vmax=300, cmap='viridis', figsize=(12, 7))